# Healthcare Clinic Dashboard Analysis

## Project Overview

This project analyses a synthetic healthcare clinic dataset using Python, Excel and Power BI.

The objective is to understand patterns in patient activity, appointments, clinical services and healthcare revenue, and to prepare a validated data model for an interactive healthcare dashboard.

### Business Question

> How are patient activity, clinical services and healthcare revenue distributed over time and across clinics, doctors and specialties?

### Tools

- Python — data inspection, cleaning and validation
- Excel — exploratory analysis and supporting calculations
- Power BI — interactive dashboard and visualisation

### Dataset

The dataset contains synthetic healthcare clinic records covering:

- Patients
- Appointments
- Doctors
- Clinics
- Payments
- Treatments
- Prescriptions
- Staff
- Patient insurance

The data is synthetic and should not be interpreted as representative of real-world healthcare activity.

### Analytical Focus

The analysis will examine:

- Appointment activity and status
- Patient utilisation
- Clinic and doctor activity
- Treatment patterns
- Prescription activity
- Insurance coverage
- Payment methods
- Treatment costs and revenue
- Trends over time

The final output will be an interactive Power BI dashboard designed to support operational and financial healthcare analysis.

## Data Sources

In [1]:
import pandas as pd
from pathlib import Path

# Project data directory
data_path = Path("../data/raw")

# Load source tables
appointments = pd.read_csv(data_path / "appointments.csv")
clinics = pd.read_csv(data_path / "clinics.csv")
doctors = pd.read_csv(data_path / "doctors.csv")
patients = pd.read_csv(data_path / "patients.csv")
payments = pd.read_csv(data_path / "payments.csv")
prescriptions = pd.read_csv(data_path / "prescriptions.csv")
staff = pd.read_csv(data_path / "staff.csv")

treatments = pd.read_csv(data_path / "treatments.csv")

insurance = pd.read_csv(
    data_path / "patient_insurance.txt",
    sep="|",
    header=None,
    names=[
        "patient_id",
        "insurance_status",
        "insurance_provider",
        "coverage_level"
    ]
)

print("Data loaded successfully.")

Data loaded successfully.


The project uses nine source files from the Kaggle Healthcare Clinic Dataset.

| Source | Records | Role |
|---|---:|---|
| `patients.csv` | 20,000 | Patient information |
| `appointments.csv` | 100,000 | Appointment activity |
| `doctors.csv` | 500 | Doctor information |
| `clinics.csv` | 50 | Clinic information |
| `payments.csv` | 100,000 | Payment transactions |
| `treatments.csv` | 100,000 | Treatment activity and costs |
| `prescriptions.csv` | 100,000 | Prescription records |
| `staff.csv` | 1,000 | Clinic staff information |
| `patient_insurance.txt` | 20,000 | Patient insurance information |

The dataset is synthetic. The records are therefore used for analytical and dashboard-development practice rather than for drawing conclusions about real-world healthcare populations or clinical outcomes.

## Data Quality Checks

Before cleaning or transforming the data, the source tables are checked for:

- Record counts
- Missing values
- Duplicate primary identifiers
- Valid relationships between tables
- Appropriate data types for dates and numeric measures

In [2]:
# Summarise the size and missing values of each source table

tables = {
    "Patients": patients,
    "Appointments": appointments,
    "Doctors": doctors,
    "Clinics": clinics,
    "Payments": payments,
    "Treatments": treatments,
    "Prescriptions": prescriptions,
    "Staff": staff,
    "Insurance": insurance
}

quality_summary = pd.DataFrame({
    "Rows": {name: len(df) for name, df in tables.items()},
    "Columns": {name: len(df.columns) for name, df in tables.items()},
    "Missing Values": {name: int(df.isna().sum().sum()) for name, df in tables.items()},
    "Duplicate Rows": {name: int(df.duplicated().sum()) for name, df in tables.items()}
})

quality_summary

,Rows,Columns,Missing Values,Duplicate Rows
Patients,20000,10,0,0
Appointments,100000,7,0,0
Doctors,500,6,0,0
Clinics,50,8,0,0
Payments,100000,6,0,0
Treatments,100000,5,0,0
Prescriptions,100000,4,0,0
Staff,1000,6,0,0
Insurance,20000,4,0,0


In [3]:
# Check uniqueness of the main identifier in each table

primary_keys = {
    "Patients": ("patient_id", patients),
    "Appointments": ("appointment_id", appointments),
    "Doctors": ("doctor_id", doctors),
    "Clinics": ("clinic_id", clinics),
    "Payments": ("payment_id", payments),
    "Treatments": ("treatment_id", treatments),
    "Prescriptions": ("prescription_id", prescriptions),
    "Staff": ("staff_id", staff),
    "Insurance": ("patient_id", insurance)
}

key_summary = pd.DataFrame([
    {
        "Table": name,
        "Primary Key": column,
        "Rows": len(df),
        "Unique Values": df[column].nunique(),
        "Duplicate Keys": int(df[column].duplicated().sum())
    }
    for name, (column, df) in primary_keys.items()
])

key_summary

,Table,Primary Key,Rows,Unique Values,Duplicate Keys
0,Patients,patient_id,20000,20000,0
1,Appointments,appointment_id,100000,100000,0
2,Doctors,doctor_id,500,500,0
3,Clinics,clinic_id,50,50,0
4,Payments,payment_id,100000,100000,0
5,Treatments,treatment_id,100000,100000,0
6,Prescriptions,prescription_id,100000,100000,0
7,Staff,staff_id,1000,1000,0
8,Insurance,patient_id,20000,20000,0


## Data Preparation

The source files contain complete records and unique primary identifiers. The next stage is to inspect date fields, categorical variables and numeric measures before creating the cleaned analytical tables.

The raw files are preserved unchanged in `data/raw/`. Any transformations will be written to `data/cleaned/`.

In [4]:
# Inspect date columns and their current data types

date_columns = {
    "Patients": ["date_of_birth", "registration_date"],
    "Appointments": ["appointment_date"],
    "Payments": ["payment_date"]
}

for table_name, columns in date_columns.items():
    df = tables[table_name]
    
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    
    for column in columns:
        print(f"\n{column}")
        print("Data type:", df[column].dtype)
        print("Example values:", df[column].head(5).tolist())


PATIENTS

date_of_birth
Data type: object
Example values: ['11/02/2011', '12/12/2002', '03/30/1939', '04/09/1984', '06/24/2020']

registration_date
Data type: object
Example values: ['10/25/2024', '05/14/2025', '12/13/2024', '07/20/2024', '06/12/2025']

APPOINTMENTS

appointment_date
Data type: object
Example values: ['07/31/2025', '03/29/2025', '07/10/2025', '06/13/2025', '12/03/2025']

PAYMENTS

payment_date
Data type: object
Example values: ['06/10/2025', '10/22/2025', '03/26/2025', '12/02/2025', '12/04/2025']


In [5]:
# Inspect the unique values in important categorical fields

categorical_columns = {
    "Appointments": ["status", "reason_for_visit"],
    "Patients": ["gender", "blood_group"],
    "Doctors": ["specialty"],
    "Clinics": ["clinic_type"],
    "Payments": ["payment_method", "payment_status"],
    "Treatments": ["treatment_type", "treatment_status"],
    "Insurance": ["insurance_status", "insurance_provider", "coverage_level"],
    "Staff": ["role"]
}

for table_name, columns in categorical_columns.items():
    df = tables[table_name]
    
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    
    for column in columns:
        print(f"\n{column}:")
        print(df[column].value_counts().to_string())


APPOINTMENTS

status:
status
Pending      33439
Completed    33409
Cancelled    33152

reason_for_visit:
reason_for_visit
Skin Allergy       16868
General Checkup    16716
Back Pain          16650
Fever              16619
Follow-up Visit    16596
Heart Pain         16551

PATIENTS

gender:
gender
Female    10122
Male       9878

blood_group:
blood_group
O+     5064
A+     5013
AB+    4984
B+     4939

DOCTORS

specialty:
specialty
Dermatology    172
Pediatrics     168
Cardiology     160

CLINICS

clinic_type:
clinic_type
Public     30
Private    20

PAYMENTS

payment_method:
payment_method
Card         33489
Cash         33482
Insurance    33029

payment_status:
payment_status
Paid    100000

TREATMENTS

treatment_type:
treatment_type
Surgery    33420
Checkup    33312
Therapy    33268

treatment_status:
treatment_status
Pending      33595
Completed    33353
Ongoing      33052

INSURANCE

insurance_status:
insurance_status
Not Insured    10057
Insured         9943

insurance_provider:


## Date Preparation

The source tables store dates as text in `MM/DD/YYYY` format. These fields are converted to pandas datetime values so that they can be used reliably for chronological analysis, monthly trends and Power BI date filtering.

The original raw files remain unchanged.

In [6]:
# Convert date fields to datetime

patients["date_of_birth"] = pd.to_datetime(
    patients["date_of_birth"],
    format="%m/%d/%Y"
)

patients["registration_date"] = pd.to_datetime(
    patients["registration_date"],
    format="%m/%d/%Y"
)

appointments["appointment_date"] = pd.to_datetime(
    appointments["appointment_date"],
    format="%m/%d/%Y"
)

payments["payment_date"] = pd.to_datetime(
    payments["payment_date"],
    format="%m/%d/%Y"
)

print("Date conversion completed.")

Date conversion completed.


In [7]:
print(patients[["date_of_birth", "registration_date"]].dtypes)
print(appointments["appointment_date"].dtype)
print(payments["payment_date"].dtype)

date_of_birth        datetime64[ns]
registration_date    datetime64[ns]
dtype: object
datetime64[ns]
datetime64[ns]


In [8]:
print("Patient registration dates:")
print(patients["registration_date"].min(), "to", patients["registration_date"].max())

print("\nAppointment dates:")
print(appointments["appointment_date"].min(), "to", appointments["appointment_date"].max())

print("\nPayment dates:")
print(payments["payment_date"].min(), "to", payments["payment_date"].max())

print("\nPatient birth dates:")
print(patients["date_of_birth"].min(), "to", patients["date_of_birth"].max())

Patient registration dates:
2024-03-22 00:00:00 to 2026-03-22 00:00:00

Appointment dates:
2025-03-22 00:00:00 to 2026-03-22 00:00:00

Payment dates:
2025-03-22 00:00:00 to 2026-03-22 00:00:00

Patient birth dates:
1910-03-28 00:00:00 to 2026-03-22 00:00:00


## Financial Data Validation

Treatment cost and payment amount are the primary financial measures in the dataset.

Before creating financial KPIs, their ranges and distributions are checked, along with the relationship between appointment, treatment and payment dates.

In [9]:
# Inspect financial measures

print("Treatment cost:")
print(treatments["treatment_cost"].describe())

print("\nPayment amount:")
print(payments["payment_amount"].describe())

print("\nPayment methods:")
print(payments["payment_method"].value_counts())

print("\nPayment status:")
print(payments["payment_status"].value_counts())

Treatment cost:
count    100000.000000
mean        801.381376
std         403.741509
min         100.000000
25%         452.367500
50%         800.600000
75%        1150.772500
max        1499.990000
Name: treatment_cost, dtype: float64

Payment amount:
count    100000.000000
mean        801.381376
std         403.741509
min         100.000000
25%         452.367500
50%         800.600000
75%        1150.772500
max        1499.990000
Name: payment_amount, dtype: float64

Payment methods:
payment_method
Card         33489
Cash         33482
Insurance    33029
Name: count, dtype: int64

Payment status:
payment_status
Paid    100000
Name: count, dtype: int64


In [10]:
# Compare appointment, treatment and payment dates

date_comparison = appointments[["appointment_id", "appointment_date"]].merge(
    payments[["appointment_id", "payment_date"]],
    on="appointment_id"
).merge(
    treatments[["appointment_id"]],
    on="appointment_id"
)

date_comparison["payment_after_appointment"] = (
    date_comparison["payment_date"] >= date_comparison["appointment_date"]
)

print("Payments on or after appointment date:",
      date_comparison["payment_after_appointment"].sum())

print("Payments before appointment date:",
      (~date_comparison["payment_after_appointment"]).sum())

Payments on or after appointment date: 49967
Payments before appointment date: 50033


## Table Structure

The following inspection records the columns available in each source table. This provides the basis for defining relationships, cleaning rules and the Power BI data model.

In [11]:
for table_name, df in tables.items():
    print(f"\n{'='*60}")
    print(table_name.upper())
    print(f"{'='*60}")
    print(df.columns.tolist())


PATIENTS
['patient_id', 'full_name', 'gender', 'date_of_birth', 'phone', 'email', 'address', 'registration_date', 'blood_group', 'emergency_contact']

APPOINTMENTS
['appointment_id', 'patient_id', 'doctor_id', 'clinic_id', 'appointment_date', 'status', 'reason_for_visit']

DOCTORS
['doctor_id', 'full_name', 'specialty', 'phone', 'email', 'clinic_id']

CLINICS
['clinic_id', 'clinic_name', 'address', 'city', 'state', 'phone', 'opening_hours', 'clinic_type']

PAYMENTS
['payment_id', 'appointment_id', 'payment_date', 'payment_amount', 'payment_method', 'payment_status']

TREATMENTS
['treatment_id', 'appointment_id', 'treatment_type', 'treatment_cost', 'treatment_status']

PRESCRIPTIONS
['prescription_id', 'appointment_id', 'medicine_name', 'instructions']

STAFF
['staff_id', 'full_name', 'role', 'clinic_id', 'phone', 'email']

INSURANCE
['patient_id', 'insurance_status', 'insurance_provider', 'coverage_level']


In [12]:
print("Appointments:", len(appointments))
print("Unique appointment IDs:", appointments["appointment_id"].nunique())

print("\nAppointments per patient:")
print(appointments["patient_id"].value_counts().describe())

print("\nAppointments per doctor:")
print(appointments["doctor_id"].value_counts().describe())

print("\nAppointments per clinic:")
print(appointments["clinic_id"].value_counts().describe())

Appointments: 100000
Unique appointment IDs: 100000

Appointments per patient:
count    19861.000000
mean         5.034993
std          2.222320
min          1.000000
25%          3.000000
50%          5.000000
75%          6.000000
max         16.000000
Name: count, dtype: float64

Appointments per doctor:
count    500.000000
mean     200.000000
std       13.940469
min      150.000000
25%      190.000000
50%      200.000000
75%      209.000000
max      243.000000
Name: count, dtype: float64

Appointments per clinic:
count      50.00000
mean     2000.00000
std        43.30292
min      1903.00000
25%      1964.50000
50%      1999.00000
75%      2037.25000
max      2081.00000
Name: count, dtype: float64


## Data Understanding Summary

The dataset contains nine relational tables covering patients, appointments, doctors, clinics, payments, treatments, prescriptions, staff and insurance.

The source tables contain complete records with no missing values or duplicate rows, and all tested primary and foreign-key relationships are valid.

The `appointments` table provides the central activity record for the dashboard. It contains 100,000 unique appointments linked to 20,000 patients, 500 doctors and 50 clinics. Most patients have multiple appointments, with an average of approximately five appointments per patient among patients with recorded appointments.

The appointment and payment activity covers 22 March 2025 to 22 March 2026. Patient registration dates begin one year earlier, while birth dates span a very wide range and include implausible synthetic edge cases.

Several categorical variables are unusually evenly distributed. This is consistent with the dataset being synthetic and means that dashboard findings should be interpreted as examples of healthcare analytics rather than evidence about real-world healthcare activity.

Payment amounts and treatment costs have identical distributions, and all payment records are marked as paid. Payment dates occur before the appointment date for approximately half of the records, so payment timing will not be interpreted as a real-world revenue-recognition measure.

The raw source files will remain unchanged. Cleaned analytical tables will be created separately for Excel and Power BI.

## Proposed Power BI Data Model

The dashboard will retain the source tables as separate entities rather than flattening the dataset into one large table.

The appointment table will act as the central activity table, with patient, doctor and clinic information providing descriptive dimensions. Payments, treatments and prescriptions will remain linked to appointments. Insurance will be linked to patients, while staff will be linked to clinics.

This structure preserves the relational nature of the dataset and reduces unnecessary duplication when building the Power BI model.